In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

JSON_PATH = "data/geocorpus-v2.json"  
MODEL_NAME = "modelicaai/albertina-base"

with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

In [3]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [4]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [5]:
id2label

{0: 'B-ambienteSedimentacao',
 1: 'B-baciaSedimentar',
 2: 'B-bentonico',
 3: 'B-campoPetrolifero',
 4: 'B-constituinteRochaSedimentar',
 5: 'B-contextoGeologicoDeBacia',
 6: 'B-elementoQuimico',
 7: 'B-eon',
 8: 'B-epoca',
 9: 'B-era',
 10: 'B-estratigrafia',
 11: 'B-estruturaGeologica',
 12: 'B-estruturaSedimentar',
 13: 'B-fosseis',
 14: 'B-geomorfologia',
 15: 'B-granulometria',
 16: 'B-idade',
 17: 'B-magmaticas',
 18: 'B-metamorficas',
 19: 'B-mineral',
 20: 'B-periodo',
 21: 'B-planctonico',
 22: 'B-procedimentoMetodologico',
 23: 'B-sedimentaresCarbonaticas',
 24: 'B-sedimentaresOrganicas',
 25: 'B-sedimentaresQuimicas',
 26: 'B-sedimentaresSiliciclasticas',
 27: 'B-sistemaPetrolifero',
 28: 'B-unidadeEstratigrafica',
 29: 'B-unidadeGeotectonica',
 30: 'I-ambienteSedimentacao',
 31: 'I-baciaSedimentar',
 32: 'I-bentonico',
 33: 'I-campoPetrolifero',
 34: 'I-constituinteRochaSedimentar',
 35: 'I-contextoGeologicoDeBacia',
 36: 'I-epoca',
 37: 'I-era',
 38: 'I-estratigrafia',
 39

In [6]:
NUM_LABELS

57

# Splits

In [7]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [8]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [9]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [10]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [11]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [12]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [16]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
# standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
# loc = loc_split(geocorpus_full)
# print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…
  7

100%|██████████| 5272/5272 [00:00<00:00, 1360510.10it/s]


semantic
reverse


# Experimentos

In [20]:
from sklearn.metrics import f1_score as skl_f1

In [21]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc"      : loc_split,
            "reverse"  : reverse_curriculum_split,
            "semantic" : semantic_cluster_split,
            "heur_len" : heur_len_split,
            "heur_rare": heur_rare_split,
            "std"      : std_split,
            "advs"     : adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []   # p/ seqeval
        flat_preds, flat_labels = [], []   # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro    = skl_f1(flat_labels, flat_preds, average="micro",    zero_division=0)
        f1_macro    = skl_f1(flat_labels, flat_preds, average="macro",    zero_division=0)
        f1_weighted = skl_f1(flat_labels, flat_preds, average="weighted", zero_division=0)

        return {
            **seqeval_metrics,            # overall_precision / recall / f1
            "f1_micro":    f1_micro,
            "f1_macro":    f1_macro,
            "f1_weighted": f1_weighted,
        }



    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none"
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [22]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "advs"]

In [23]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geocorpus_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([57]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([57, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1054/1054 [00:00<00:00, 11407.18 examples/s]
/tmp/ipykernel_144877/2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/n

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6587085052451003
F1 Micro: 0.9887927208166889
F1 Weighted: 0.9884786309189074
{'eval_loss': 0.05377170071005821, 'eval_ambienteSedimentacao': {'precision': 0.6, 'recall': 0.5, 'f1': 0.5454545454545454, 'number': 6}, 'eval_baciaSedimentar': {'precision': 0.7543859649122807, 'recall': 0.8269230769230769, 'f1': 0.7889908256880734, 'number': 52}, 'eval_bentonico': {'precision': 0.6666666666666666, 'recall': 0.36363636363636365, 'f1': 0.4705882352941177, 'number': 11}, 'eval_campoPetrolifero': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}, 'eval_constituinteRochaSedimentar': {'precision': 0.75, 'recall': 1.0, 'f1': 0.8571428571428571, 'number': 3}, 'eval_contextoGeologicoDeBacia': {'precision': 0.8131868131868132, 'recall': 0.74, 'f1': 0.774869109947644, 'number': 100}, 'eval_elementoQuimico': {'precision': 1.0, 'recall': 0.7142857142857143, 'f1': 0.8333333333333333, 'number': 7}, 'eval_eon': {'precision': 0.9032258064516129, 'recall': 0.9655172413793104, 'f1': 0.933

340

In [24]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [25]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geocorpus_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([57]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([57, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1054/1054 [00:00<00:00, 6728.66 examples/s]
/tmp/ipykernel_144877/2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Ambientesedimentacao,Baciasedimentar,Constituinterochasedimentar,Contextogeologicodebacia,Elementoquimico,Eon,Epoca,Era,Estratigrafia,Estruturageologica,Estruturasedimentar,Fosseis,Geomorfologia,Granulometria,Idade,Magmaticas,Metamorficas,Mineral,Periodo,Planctonico,Procedimentometodologico,Sedimentarescarbonaticas,Sedimentaresorganicas,Sedimentaresquimicas,Sedimentaressiliciclasticas,Sistemapetrolifero,Unidadeestratigrafica,Unidadegeotectonica,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.190119,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 22}","{'precision': 0.6166666666666667, 'recall': 0.5522388059701493, 'f1': 0.5826771653543308, 'number': 67}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 26}","{'precision': 0.620253164556962, 'recall': 0.6282051282051282, 'f1': 0.6242038216560509, 'number': 78}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.9285714285714286, 'recall': 0.9285714285714286, 'f1': 0.9285714285714286, 'number': 56}","{'precision': 0.7857142857142857, 'recall': 0.7129629629629629, 'f1': 0.7475728155339806, 'number': 108}","{'precision': 0.9397590361445783, 'recall': 0.9512195121951219, 'f1': 0.9454545454545454, 'number': 82}","{'precision': 0.9705882352941176, 'recall': 0.9166666666666666, 'f1': 0.9428571428571428, 'number': 36}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 6}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 21}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 18}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 25}","{'precision': 0.8529411764705882, 'recall': 0.8721804511278195, 'f1': 0.862453531598513, 'number': 133}","{'precision': 0.7454545454545455, 'recall': 0.6119402985074627, 'f1': 0.6721311475409837, 'number': 67}","{'precision': 0.8928571428571429, 'recall': 0.49019607843137253, 'f1': 0.6329113924050633, 'number': 51}","{'precision': 0.8125, 'recall': 0.8666666666666667, 'f1': 0.8387096774193549, 'number': 30}","{'precision': 0.725, 'recall': 0.9206349206349206, 'f1': 0.8111888111888111, 'number': 126}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 18}","{'precision': 1.0, 'recall': 0.6086956521739131, 'f1': 0.7567567567567568, 'number': 23}","{'precision': 0.5319148936170213, 'recall': 0.49019607843137253, 'f1': 0.5102040816326531, 'number': 51}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 0.7514792899408284, 'recall': 0.8355263157894737, 'f1': 0.7912772585669782, 'number': 152}","{'precision': 0.6923076923076923, 'recall': 0.3103448275862069, 'f1': 0.4285714285714286, 'number': 29}","{'precision': 0.581081081081081, 'recall': 0.4942528735632184, 'f1': 0.5341614906832297, 'number': 87}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.762742,0.655589,0.705118,0.965121,0.965121,0.313011,0.956652
2,No log,0.139391,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 22}","{'precision': 0.76, 'recall': 0.8507462686567164, 'f1': 0.8028169014084507, 'number': 67}","{'precision': 0.9230769230769231, 'recall': 0.46153846153846156, 'f1': 0.6153846153846155, 'number': 26}","{'precision': 0.6547619047619048, 'recall': 0.7051282051282052, 'f1': 0.6790123456790124, 'number': 78}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.9642857142857143, 'recall': 0.9642857142857143, 'f1': 0.9642857142857143, 'number': 56}","{'precision': 0.8947368421052632, 'recall': 0.9444444444444444, 'f1': 0.918918918918919, 'number': 108}","{'precision': 0.9195402298850575, 'recall': 0.975609756097561, 'f1': 0.9467455621301775, 'number': 82}","{'precision': 0.9705882352941176, 'recall': 0.9166666666666666, 'f1': 0.9428571428571428, 'number': 36}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.3899700393069398
F1 Micro: 0.9428287050860954
F1 Weighted: 0.9292042647934189
{'eval_loss': 0.4567084014415741, 'eval_ambienteSedimentacao': {'precision': 1.0, 'recall': 0.008928571428571428, 'f1': 0.017699115044247787, 'number': 112}, 'eval_baciaSedimentar': {'precision': 0.7262357414448669, 'recall': 0.6221498371335505, 'f1': 0.6701754385964912, 'number': 307}, 'eval_bentonico': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 27}, 'eval_campoPetrolifero': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}, 'eval_constituinteRochaSedimentar': {'precision': 0.96, 'recall': 0.42857142857142855, 'f1': 0.5925925925925924, 'number': 56}, 'eval_contextoGeologicoDeBacia': {'precision': 0.5542521994134897, 'recall': 0.5869565217391305, 'f1': 0.5701357466063349, 'number': 322}, 'eval_elementoQuimico': {'precision': 1.0, 'recall': 0.058823529411764705, 'f1': 0.1111111111111111, 'number': 17}, 'eval_eon': {'precision': 0.881578947368421, 'recall': 0.8589743589743589, 'f

0

In [26]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [27]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geocorpus_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


100%|██████████| 5272/5272 [00:00<00:00, 1341525.86it/s]
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([57]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([57, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1081/1081 [00:00<00:00, 11132.52 examples/s]
/tmp/ipykernel_144877/2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/n

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6668808184213878
F1 Micro: 0.9898943349023996
F1 Weighted: 0.9897224714981325
{'eval_loss': 0.04663388803601265, 'eval_ambienteSedimentacao': {'precision': 0.6666666666666666, 'recall': 0.5, 'f1': 0.5714285714285715, 'number': 4}, 'eval_baciaSedimentar': {'precision': 0.926829268292683, 'recall': 1.0, 'f1': 0.9620253164556963, 'number': 38}, 'eval_bentonico': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}, 'eval_campoPetrolifero': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1}, 'eval_constituinteRochaSedimentar': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}, 'eval_contextoGeologicoDeBacia': {'precision': 0.5135135135135135, 'recall': 0.9047619047619048, 'f1': 0.6551724137931034, 'number': 21}, 'eval_elementoQuimico': {'precision': 0.6666666666666666, 'recall': 1.0, 'f1': 0.8, 'number': 2}, 'eval_eon': {'precision': 0.9562043795620438, 'recall': 0.9776119402985075, 'f1': 0.966789667896679, 'number': 134}, 'eval_epoca': {'precision': 0.86792

0

In [28]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [29]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geocorpus_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([57]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([57, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1054/1054 [00:00<00:00, 9412.04 examples/s]
/tmp/ipykernel_144877/2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Ambientesedimentacao,Baciasedimentar,Bentonico,Constituinterochasedimentar,Contextogeologicodebacia,Elementoquimico,Eon,Epoca,Era,Estratigrafia,Estruturageologica,Estruturasedimentar,Fosseis,Geomorfologia,Granulometria,Idade,Magmaticas,Metamorficas,Mineral,Periodo,Planctonico,Procedimentometodologico,Sedimentarescarbonaticas,Sedimentaresorganicas,Sedimentaresquimicas,Sedimentaressiliciclasticas,Sistemapetrolifero,Unidadeestratigrafica,Unidadegeotectonica,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.126025,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 13}","{'precision': 0.6909090909090909, 'recall': 0.8085106382978723, 'f1': 0.7450980392156863, 'number': 47}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}","{'precision': 1.0, 'recall': 0.06666666666666667, 'f1': 0.125, 'number': 15}","{'precision': 0.6666666666666666, 'recall': 0.6, 'f1': 0.631578947368421, 'number': 60}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.8636363636363636, 'recall': 0.59375, 'f1': 0.7037037037037037, 'number': 32}","{'precision': 0.7349397590361446, 'recall': 0.8243243243243243, 'f1': 0.7770700636942676, 'number': 74}","{'precision': 0.75, 'recall': 0.9545454545454546, 'f1': 0.84, 'number': 44}","{'precision': 0.8, 'recall': 0.6666666666666666, 'f1': 0.7272727272727272, 'number': 24}","{'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}","{'precision': 1.0, 'recall': 0.1111111111111111, 'f1': 0.19999999999999998, 'number': 9}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 4}","{'precision': 0.75, 'recall': 0.6, 'f1': 0.6666666666666665, 'number': 15}","{'precision': 0.6724137931034483, 'recall': 0.9285714285714286, 'f1': 0.78, 'number': 84}","{'precision': 0.7435897435897436, 'recall': 0.5272727272727272, 'f1': 0.6170212765957446, 'number': 55}","{'precision': 1.0, 'recall': 0.5428571428571428, 'f1': 0.7037037037037037, 'number': 35}","{'precision': 0.75, 'recall': 0.6428571428571429, 'f1': 0.6923076923076924, 'number': 14}","{'precision': 0.875, 'recall': 0.7, 'f1': 0.7777777777777777, 'number': 80}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 1.0, 'recall': 0.8461538461538461, 'f1': 0.9166666666666666, 'number': 13}","{'precision': 0.5483870967741935, 'recall': 0.5483870967741935, 'f1': 0.5483870967741935, 'number': 31}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 0.7751937984496124, 'recall': 0.847457627118644, 'f1': 0.8097165991902834, 'number': 118}","{'precision': 0.75, 'recall': 0.5, 'f1': 0.6, 'number': 6}","{'precision': 0.6527777777777778, 'recall': 0.746031746031746, 'f1': 0.6962962962962964, 'number': 63}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}",0.733170,0.694090,0.713095,0.972015,0.972015,0.412162,0.968361
2,No log,0.087777,"{'precision': 0.6363636363636364, 'recall': 0.5384615384615384, 'f1': 0.5833333333333334, 'number': 13}","{'precision': 0.6923076923076923, 'recall': 0.7659574468085106, 'f1': 0.7272727272727273, 'number': 47}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}","{'precision': 0.8461538461538461, 'recall': 0.7333333333333333, 'f1': 0.7857142857142856, 'number': 15}","{'precision': 0.6571428571428571, 'recall': 0.7666666666666667, 'f1': 0.7076923076923077, 'number': 60}","{'precision': 1.0, 'recall': 0.2, 'f1': 0.33333333333333337, 'number': 5}","{'precision': 0.9666666666666667, 'recall': 0.90625, 'f1': 0.9354838709677419, 'number': 32}","{'precision': 0.8809523809523809, 'recall': 1.0, 'f1': 0.9367088607594937, 'number': 74}","{'precision': 0.9565217391304348, 'recall': 1.0, 'f1': 0.9777777777777777, 'number': 44}","{'precision': 0.8, 'recall': 0.6666666666666666, 'f1': 0.7272727272727272, 'number': 24}","{'precision': 0.75, 'recall':

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6796626114892569
F1 Micro: 0.985896976216087
F1 Weighted: 0.9856486616592398
{'eval_loss': 0.0655684620141983, 'eval_ambienteSedimentacao': {'precision': 0.7368421052631579, 'recall': 0.717948717948718, 'f1': 0.7272727272727273, 'number': 39}, 'eval_baciaSedimentar': {'precision': 0.8380952380952381, 'recall': 0.8380952380952381, 'f1': 0.8380952380952381, 'number': 105}, 'eval_bentonico': {'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8, 'number': 3}, 'eval_campoPetrolifero': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}, 'eval_constituinteRochaSedimentar': {'precision': 0.8636363636363636, 'recall': 0.8636363636363636, 'f1': 0.8636363636363636, 'number': 22}, 'eval_contextoGeologicoDeBacia': {'precision': 0.7549668874172185, 'recall': 0.8444444444444444, 'f1': 0.7972027972027972, 'number': 135}, 'eval_elementoQuimico': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1}, 'eval_eon': {'precision': 0.9818181818181818, 'recall': 0.9642857142857143,

320

In [30]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [31]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geocorpus_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([57]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([57, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1054/1054 [00:00<00:00, 9022.68 examples/s]
/tmp/ipykernel_144877/2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Ambientesedimentacao,Baciasedimentar,Bentonico,Campopetrolifero,Constituinterochasedimentar,Contextogeologicodebacia,Elementoquimico,Eon,Epoca,Era,Estratigrafia,Estruturageologica,Estruturasedimentar,Fosseis,Geomorfologia,Granulometria,Idade,Magmaticas,Metamorficas,Mineral,Periodo,Planctonico,Procedimentometodologico,Sedimentarescarbonaticas,Sedimentaresorganicas,Sedimentaresquimicas,Sedimentaressiliciclasticas,Sistemapetrolifero,Unidadeestratigrafica,Unidadegeotectonica,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.148016,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.6451612903225806, 'recall': 0.6666666666666666, 'f1': 0.6557377049180327, 'number': 60}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 18}","{'precision': 0.47619047619047616, 'recall': 0.5882352941176471, 'f1': 0.5263157894736842, 'number': 68}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}","{'precision': 0.9375, 'recall': 0.7142857142857143, 'f1': 0.8108108108108109, 'number': 21}","{'precision': 0.6363636363636364, 'recall': 0.6621621621621622, 'f1': 0.6490066225165563, 'number': 74}","{'precision': 0.7209302325581395, 'recall': 0.9393939393939394, 'f1': 0.8157894736842105, 'number': 33}","{'precision': 0.7692307692307693, 'recall': 0.75, 'f1': 0.7594936708860761, 'number': 40}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 12}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 7}","{'precision': 1.0, 'recall': 0.2, 'f1': 0.33333333333333337, 'number': 20}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 7}","{'precision': 0.5555555555555556, 'recall': 0.5263157894736842, 'f1': 0.5405405405405405, 'number': 19}","{'precision': 0.6923076923076923, 'recall': 0.9642857142857143, 'f1': 0.8059701492537313, 'number': 84}","{'precision': 0.8125, 'recall': 0.6341463414634146, 'f1': 0.7123287671232876, 'number': 41}","{'precision': 0.7, 'recall': 0.5833333333333334, 'f1': 0.6363636363636365, 'number': 24}","{'precision': 0.6666666666666666, 'recall': 0.8421052631578947, 'f1': 0.744186046511628, 'number': 19}","{'precision': 0.7654320987654321, 'recall': 0.7948717948717948, 'f1': 0.7798742138364779, 'number': 78}","{'precision': 1.0, 'recall': 0.1111111111111111, 'f1': 0.19999999999999998, 'number': 9}","{'precision': 0.9285714285714286, 'recall': 0.7647058823529411, 'f1': 0.8387096774193549, 'number': 17}","{'precision': 0.4807692307692308, 'recall': 0.6578947368421053, 'f1': 0.5555555555555556, 'number': 38}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.7214285714285714, 'recall': 0.8145161290322581, 'f1': 0.7651515151515151, 'number': 124}","{'precision': 0.3333333333333333, 'recall': 0.3333333333333333, 'f1': 0.3333333333333333, 'number': 6}","{'precision': 0.6373626373626373, 'recall': 0.725, 'f1': 0.6783625730994152, 'number': 80}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.673491,0.672043,0.672766,0.965703,0.965703,0.388005,0.960477
2,No log,0.094702,"{'precision': 0.75, 'recall': 0.6428571428571429, 'f1': 0.6923076923076924, 'number': 14}","{'precision': 0.703125, 'recall': 0.75, 'f1': 0.7258064516129032, 'number': 60}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 0.875, 'recall': 0.3888888888888889, 'f1': 0.5384615384615385, 'number': 18}","{'precision': 0.7230769230769231, 'recall': 0.6911764705882353, 'f1': 0.7067669172932332, 'number': 68}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}","{'precision': 0.9545454545454546, 'recall': 1.0, 'f1': 0.9767441860465117, 'number': 21}","{'precision': 0.8023255813953488, 'recall': 0.9324324324324325, 'f1': 0.8625, 'number'

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6523659634606951
F1 Micro: 0.9844875196274839
F1 Weighted: 0.9843165729452448
{'eval_loss': 0.07353035360574722, 'eval_ambienteSedimentacao': {'precision': 0.6451612903225806, 'recall': 0.6451612903225806, 'f1': 0.6451612903225806, 'number': 31}, 'eval_baciaSedimentar': {'precision': 0.828125, 'recall': 0.8983050847457628, 'f1': 0.8617886178861789, 'number': 118}, 'eval_bentonico': {'precision': 0.5, 'recall': 0.5, 'f1': 0.5, 'number': 2}, 'eval_campoPetrolifero': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}, 'eval_constituinteRochaSedimentar': {'precision': 0.8125, 'recall': 0.7222222222222222, 'f1': 0.7647058823529411, 'number': 36}, 'eval_contextoGeologicoDeBacia': {'precision': 0.6492146596858639, 'recall': 0.8104575163398693, 'f1': 0.7209302325581396, 'number': 153}, 'eval_elementoQuimico': {'precision': 0.8888888888888888, 'recall': 0.8888888888888888, 'f1': 0.8888888888888888, 'number': 9}, 'eval_eon': {'precision': 0.8809523809523809, 'recall': 0.880952

320

In [25]:
del results, trainer_all#, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

NameError: name 'results' is not defined

In [23]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geocorpus_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([57]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([57, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1054/1054 [00:00<00:00, 8299.64 examples/s]
/tmp/ipykernel_163479/2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Ambientesedimentacao,Baciasedimentar,Bentonico,Campopetrolifero,Constituinterochasedimentar,Contextogeologicodebacia,Eon,Epoca,Era,Estratigrafia,Estruturageologica,Estruturasedimentar,Fosseis,Geomorfologia,Granulometria,Idade,Magmaticas,Metamorficas,Mineral,Periodo,Planctonico,Procedimentometodologico,Sedimentarescarbonaticas,Sedimentaresorganicas,Sedimentaresquimicas,Sedimentaressiliciclasticas,Sistemapetrolifero,Unidadeestratigrafica,Unidadegeotectonica,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.124135,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 10}","{'precision': 0.6181818181818182, 'recall': 0.7555555555555555, 'f1': 0.6799999999999999, 'number': 45}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.5573770491803278, 'recall': 0.5573770491803278, 'f1': 0.5573770491803278, 'number': 61}","{'precision': 0.8275862068965517, 'recall': 0.9230769230769231, 'f1': 0.8727272727272727, 'number': 26}","{'precision': 0.7752808988764045, 'recall': 0.92, 'f1': 0.8414634146341463, 'number': 75}","{'precision': 0.725, 'recall': 0.9354838709677419, 'f1': 0.8169014084507041, 'number': 31}","{'precision': 0.6923076923076923, 'recall': 0.5625, 'f1': 0.6206896551724138, 'number': 16}","{'precision': 0.5, 'recall': 0.6, 'f1': 0.5454545454545454, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 7}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 4}","{'precision': 0.5, 'recall': 0.4, 'f1': 0.4444444444444445, 'number': 5}","{'precision': 0.7227722772277227, 'recall': 0.9358974358974359, 'f1': 0.8156424581005586, 'number': 78}","{'precision': 0.6744186046511628, 'recall': 0.9560439560439561, 'f1': 0.7909090909090909, 'number': 91}","{'precision': 0.8214285714285714, 'recall': 0.48936170212765956, 'f1': 0.6133333333333333, 'number': 47}","{'precision': 0.875, 'recall': 0.9545454545454546, 'f1': 0.9130434782608695, 'number': 22}","{'precision': 0.6767676767676768, 'recall': 0.8589743589743589, 'f1': 0.7570621468926554, 'number': 78}","{'precision': 0.25, 'recall': 0.1111111111111111, 'f1': 0.15384615384615383, 'number': 9}","{'precision': 1.0, 'recall': 0.8333333333333334, 'f1': 0.9090909090909091, 'number': 12}","{'precision': 0.5333333333333333, 'recall': 0.48484848484848486, 'f1': 0.507936507936508, 'number': 33}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 0.801980198019802, 'recall': 0.8526315789473684, 'f1': 0.826530612244898, 'number': 95}","{'precision': 0.7142857142857143, 'recall': 0.8333333333333334, 'f1': 0.7692307692307692, 'number': 6}","{'precision': 0.6461538461538462, 'recall': 0.7924528301886793, 'f1': 0.711864406779661, 'number': 53}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.705556,0.762305,0.732833,0.972239,0.972239,0.440353,0.969273
2,No log,0.092751,"{'precision': 1.0, 'recall': 0.6, 'f1': 0.7499999999999999, 'number': 10}","{'precision': 0.7307692307692307, 'recall': 0.8444444444444444, 'f1': 0.7835051546391751, 'number': 45}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}","{'precision': 0.46296296296296297, 'recall': 0.819672131147541, 'f1': 0.591715976331361, 'number': 61}","{'precision': 0.8666666666666667, 'recall': 1.0, 'f1': 0.9285714285714286, 'number': 26}","{'precision': 0.8902439024390244, 'recall': 0.9733333333333334, 'f1': 0.9299363057324841, 'number': 75}","{'precision': 0.7209302325581395, 'recall': 1.0, 'f1': 0.8378378378378378, 'number': 31}","{'precision': 0.7142857142857143, 'recall': 0.625, 'f1': 0.6666666666666666, 'number': 16

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.6020732383933304
F1 Micro: 0.9723849372384937
F1 Weighted: 0.9686079379785297
{'eval_loss': 0.15635041892528534, 'eval_ambienteSedimentacao': {'precision': 0.30120481927710846, 'recall': 0.27472527472527475, 'f1': 0.28735632183908044, 'number': 91}, 'eval_baciaSedimentar': {'precision': 0.8033707865168539, 'recall': 0.8218390804597702, 'f1': 0.8125, 'number': 174}, 'eval_bentonico': {'precision': 0.4, 'recall': 0.3333333333333333, 'f1': 0.3636363636363636, 'number': 6}, 'eval_constituinteRochaSedimentar': {'precision': 0.9047619047619048, 'recall': 0.4470588235294118, 'f1': 0.5984251968503936, 'number': 85}, 'eval_contextoGeologicoDeBacia': {'precision': 0.7272727272727273, 'recall': 0.7619047619047619, 'f1': 0.7441860465116279, 'number': 210}, 'eval_elementoQuimico': {'precision': 0.8, 'recall': 1.0, 'f1': 0.888888888888889, 'number': 4}, 'eval_eon': {'precision': 0.8695652173913043, 'recall': 0.9090909090909091, 'f1': 0.888888888888889, 'number': 22}, 'eval_epoca': {'prec

20

In [24]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()